# BERT fine-tuning with synthetic mountain examples

This pipeline appends `synthetic_processed.json` to the original train split only. Validation and test stay unchanged, so final metrics remain comparable with the baseline.

## 1. Configuration

Set the BERT training hyperparameters here. The default `weighted_cross_entropy` is appropriate because BIO labels are imbalanced: most tokens are `O`.

In [ ]:
import os
import shutil
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
_candidates = sorted({path.parent for path in INPUT_ROOT.rglob('requirements.txt') if path.parent.name == 'ner'}, key=str) if INPUT_ROOT.is_dir() else []
KAGGLE_NER_ROOT = _candidates[0] if _candidates else None
IN_KAGGLE = KAGGLE_NER_ROOT is not None
PROJECT_ROOT = Path('/kaggle/working') if IN_KAGGLE else Path(r'D:\quantum_task')
WORK_NER_ROOT = PROJECT_ROOT / 'src/ner'
if IN_KAGGLE:
    shutil.copytree(KAGGLE_NER_ROOT, WORK_NER_ROOT, dirs_exist_ok=True)
os.chdir(PROJECT_ROOT)

data_candidates = sorted([path.parent for path in Path('/kaggle/input').rglob('few_nerd_mountains_output') if path.is_dir()], key=str) if Path('/kaggle/input').is_dir() else []
DATA_ROOT = data_candidates[0] if data_candidates else WORK_NER_ROOT / 'data'
BASE_DATA = DATA_ROOT / 'few_nerd_mountains_output'
SYNTHETIC_JSON = DATA_ROOT / 'synthetic_processed.json'
RUNTIME_ROOT = Path('/kaggle/working') if IN_KAGGLE else PROJECT_ROOT / 'outputs'
AUGMENTED_DATA = RUNTIME_ROOT / 'few_nerd_mountains_augmented'
RUN_DIR = RUNTIME_ROOT / 'bert_synthetic_run'
CHECKPOINT = RUN_DIR / 'checkpoint'

MODEL_NAME = 'bert-base-cased'
LOSS_NAME = 'weighted_cross_entropy'
LEARNING_RATE = 2e-5
EPOCHS = 5
TRAIN_BATCH_SIZE = 16
MAX_LENGTH = 256
DEVICE = 'auto'

if not BASE_DATA.is_dir() or not SYNTHETIC_JSON.is_file():
    raise FileNotFoundError(f'NER data not found under {DATA_ROOT}')
print(f'project_root={PROJECT_ROOT}')
print(f'working_ner_root={WORK_NER_ROOT}')
print(f'base_data={BASE_DATA}')
print(f'synthetic_json={SYNTHETIC_JSON}')
print(f'augmented_data={AUGMENTED_DATA}')


## 2. Install BERT dependencies

Only the core BERT dependencies are installed. The archive no longer requires `evaluate` or `seqeval`; BIO metrics are computed by the project code.

In [ ]:
!python -m pip install -q -r src/ner/requirements.txt


## 3. Load and validate the source data

Both sources must use the same schema: `sentence`, word-level `tokens`, and BIO ids where `0=O`, `1=B-Mountain`, and `2=I-Mountain`. Invalid synthetic records stop the notebook before training.

In [ ]:
import json
from datasets import load_from_disk

base_dataset = load_from_disk(str(BASE_DATA))
synthetic_records = json.loads(SYNTHETIC_JSON.read_text(encoding='utf-8'))

required_columns = {'sentence', 'tokens', 'labels'}
assert required_columns.issubset(base_dataset['train'].column_names)
assert isinstance(synthetic_records, list) and synthetic_records

for index, record in enumerate(synthetic_records):
    assert required_columns.issubset(record), f'missing columns in synthetic record {index}'
    assert len(record['tokens']) == len(record['labels']), f'token/label mismatch in synthetic record {index}'
    assert set(record['labels']).issubset({0, 1, 2}), f'invalid BIO ids in synthetic record {index}'

print(f"base_train={len(base_dataset['train'])}")
print(f"base_validation={len(base_dataset['validation'])}")
print(f"base_test={len(base_dataset['test'])}")
print(f'synthetic_train={len(synthetic_records)}')

## 4. Create the augmented DatasetDict

The synthetic records receive unique ids and are concatenated with the original train split. Validation and test are copied unchanged, preventing synthetic data from contaminating model selection or final evaluation.

In [ ]:
import shutil
from datasets import Dataset, DatasetDict, concatenate_datasets

synthetic_dataset = Dataset.from_list(synthetic_records)
synthetic_dataset = synthetic_dataset.add_column('id', [f'synthetic-{index:05d}' for index in range(len(synthetic_dataset))])
synthetic_dataset = synthetic_dataset.select_columns(base_dataset['train'].column_names)

augmented_dataset = DatasetDict({
    'train': concatenate_datasets([base_dataset['train'], synthetic_dataset]),
    'validation': base_dataset['validation'],
    'test': base_dataset['test'],
})

if AUGMENTED_DATA.exists():
    shutil.rmtree(AUGMENTED_DATA)
augmented_dataset.save_to_disk(str(AUGMENTED_DATA))

metadata = {
    'base_train_examples': len(base_dataset['train']),
    'synthetic_examples': len(synthetic_dataset),
    'augmented_train_examples': len(augmented_dataset['train']),
    'validation_examples': len(augmented_dataset['validation']),
    'test_examples': len(augmented_dataset['test']),
}
(AUGMENTED_DATA / 'augmentation_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(metadata)

## 5. Fine-tune BERT

This calls the project training script with the augmented DatasetDict. It trains on the combined train split, selects the checkpoint by validation F1, and preserves the test split for one final evaluation.

In [ ]:
!python src/ner/fine_tuning/train_bert.py --data-path "{AUGMENTED_DATA}" --model-name "{MODEL_NAME}" --loss-name "{LOSS_NAME}" --learning-rate {LEARNING_RATE} --num-epochs {EPOCHS} --train-batch-size {TRAIN_BATCH_SIZE} --max-length {MAX_LENGTH} --output-dir "{RUN_DIR / 'trainer_output'}" --save-model-path "{CHECKPOINT}" --device "{DEVICE}" --disable-wandb

## 6. Final test evaluation and one-sentence inference

The selected checkpoint is evaluated on the untouched test split. The final command also prints token predictions for a sample sentence, which is useful for a qualitative report check.

In [ ]:
!python src/ner/fine_tuning/bert_experiments.py --model-path "{CHECKPOINT}" --data-path "{AUGMENTED_DATA}" --evaluation-split test --sentence "The expedition climbed Mount Everest during winter." --device "{DEVICE}" --disable-wandb

## Output

`few_nerd_mountains_augmented` contains the reproducible combined DatasetDict and metadata. `bert_synthetic_run/checkpoint` contains the trained BERT model, tokenizer, and config. Compare its test F1 with the baseline trained without the 196 synthetic records.

## 7. Training curves

The chart is generated from the saved Hugging Face `trainer_state.json`. It plots training loss, validation loss, validation F1, precision, and recall, and marks the validation-selected checkpoint.

In [ ]:
!python src/ner/utils/plot_training_curves.py --state-path "{RUN_DIR / 'trainer_output' / 'checkpoint-2550' / 'trainer_state.json'}" --output "{RUN_DIR / 'training_curves.png'}"

from IPython.display import Image, display
display(Image(filename=str(RUN_DIR / 'training_curves.png')))